<a href="https://colab.research.google.com/github/CJR07/Algoritmos/blob/main/Proyecto_final_Carlos_Rebellon_ipyn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
from dataclasses import dataclass


# =====================================================
#                      CLASE JUGADOR
# =====================================================

@dataclass
class Jugador:
    capital_inicial: float
    apuesta: float
    prob_ganar: float
    tipo_utilidad: str = "sqrt"  # "lineal", "log", "sqrt"

    def utilidad(self, riqueza: float) -> float:
        """
        Función de utilidad del jugador (modelo de aversión al riesgo).
        """
        if riqueza <= 0:
            # Utilidad muy negativa si se arruina
            return -10_000.0

        if self.tipo_utilidad == "lineal":
            return riqueza
        elif self.tipo_utilidad == "log":
            return math.log(riqueza)
        elif self.tipo_utilidad == "sqrt":
            return math.sqrt(riqueza)
        # Por defecto, raíz cuadrada
        return math.sqrt(riqueza)


# =====================================================
#                      CLASE SEGURO
# =====================================================

@dataclass
class Seguro:
    prima: float         # Pago inicial por contratar seguro
    cobertura: float     # Porcentaje [0,1] de la pérdida cubierta

    def cobrar_prima(self, capital: float) -> float:
        """
        Descuenta la prima del capital al inicio del juego.
        """
        return capital - self.prima

    def aplicar_cobertura(self, perdida: float) -> float:
        """
        Devuelve la pérdida efectiva que sufre el jugador
        después de aplicar la cobertura del seguro.
        """
        return perdida * (1.0 - self.cobertura)


# =====================================================
#                  CLASE SIMULACIÓN
# =====================================================

class SimulacionRuina:
    def __init__(
        self,
        jugador: Jugador,
        seguro: Seguro | None = None,
        max_rondas: int = 1000,
        n_simulaciones: int = 10_000
    ) -> None:
        self.jugador = jugador
        self.seguro = seguro
        self.max_rondas = max_rondas
        self.n_simulaciones = n_simulaciones

    # --------------- MÉTODOS INTERNOS -----------------

    def _simular_una_vez(self, usar_seguro: bool = False) -> tuple[float, bool]:
        """
        Simula una trayectoria de juego.
        Regresa:
            capital_final (float)
            arruinado (bool)
        """
        capital = self.jugador.capital_inicial

        # Prima del seguro al inicio
        if usar_seguro and self.seguro is not None:
            capital = self.seguro.cobrar_prima(capital)

        for _ in range(self.max_rondas):
            if capital <= 0:
                return 0.0, True

            # No puede seguir apostando → ruina práctica
            if capital < self.jugador.apuesta:
                return capital, True

            # Se juega la ronda
            if random.random() < self.jugador.prob_ganar:
                # Gana
                capital += self.jugador.apuesta
            else:
                # Pierde
                perdida = self.jugador.apuesta
                if usar_seguro and self.seguro is not None:
                    perdida = self.seguro.aplicar_cobertura(perdida)
                capital -= perdida

            if capital <= 0:
                return 0.0, True

        # Sobrevive hasta el final de las rondas
        return capital, capital <= 0

    def _estadisticas(self, usar_seguro: bool = False) -> tuple[float, float, float]:
        """
        Corre muchas simulaciones para estimar:
            - probabilidad de ruina
            - utilidad esperada final
            - capital esperado final
        """
        ruinas = 0
        suma_utilidad = 0.0
        suma_capital = 0.0

        for _ in range(self.n_simulaciones):
            capital_final, arruinado = self._simular_una_vez(usar_seguro=usar_seguro)
            if arruinado:
                ruinas += 1

            suma_capital += capital_final
            suma_utilidad += self.jugador.utilidad(capital_final)

        prob_ruina = ruinas / self.n_simulaciones
        utilidad_esperada = suma_utilidad / self.n_simulaciones
        capital_esperado = suma_capital / self.n_simulaciones

        return prob_ruina, utilidad_esperada, capital_esperado

    # --------------- MÉTODO PRINCIPAL -----------------

    def comparar_sin_y_con_seguro(self) -> None:
        """
        Compara los escenarios:
            - Sin seguro
            - Con seguro
        Imprime resultados y un análisis de cuándo conviene el seguro.
        """
        barra = "=" * 60
        print(barra)
        print("      SIMULACIÓN DE RUINA CON Y SIN SEGURO".center(60))
        print(barra)
        print(f"Capital inicial:          {self.jugador.capital_inicial:10.2f}")
        print(f"Apuesta por ronda:        {self.jugador.apuesta:10.2f}")
        print(f"Prob. de ganar por ronda: {self.jugador.prob_ganar:10.2%}")
        print(f"Función de utilidad:      {self.jugador.tipo_utilidad}")
        print(f"N° de simulaciones:       {self.n_simulaciones}")
        print(f"Máx. rondas por juego:    {self.max_rondas}")
        print(barra)

        # ------ ESCENARIO SIN SEGURO ------
        print("\n>>> ESCENARIO SIN SEGURO")
        p_ruina_sin, u_esp_sin, cap_esp_sin = self._estadisticas(usar_seguro=False)
        print(f"Probabilidad de ruina:     {p_ruina_sin:10.4%}")
        print(f"Capital esperado final:    {cap_esp_sin:10.2f}")
        print(f"Utilidad esperada final:   {u_esp_sin:10.4f}")

        if self.seguro is None:
            print("\nNo se definió un seguro. No hay comparación posible.")
            return

        # ------ ESCENARIO CON SEGURO ------
        print("\n>>> ESCENARIO CON SEGURO")
        print(f"Prima del seguro:          {self.seguro.prima:10.2f}")
        print(f"Cobertura de pérdidas:     {self.seguro.cobertura:10.2%}")
        p_ruina_con, u_esp_con, cap_esp_con = self._estadisticas(usar_seguro=True)
        print(f"Probabilidad de ruina:     {p_ruina_con:10.4%}")
        print(f"Capital esperado final:    {cap_esp_con:10.2f}")
        print(f"Utilidad esperada final:   {u_esp_con:10.4f}")

        # ------ ANÁLISIS ------
        print("\n" + barra)
        print("              ANÁLISIS DE DECISIÓN".center(60))
        print(barra)

        # Cambios en probabilidad de ruina
        if p_ruina_con < p_ruina_sin:
            print(f"- El seguro REDUCE la probabilidad de ruina "
                  f"de {p_ruina_sin:.4%} a {p_ruina_con:.4%}.")
        elif p_ruina_con > p_ruina_sin:
            print(f"- El seguro AUMENTA la probabilidad de ruina "
                  f"de {p_ruina_sin:.4%} a {p_ruina_con:.4%} (parámetros poco realistas).")
        else:
            print("- El seguro no modifica la probabilidad de ruina.")

        # Cambios en utilidad esperada
        if u_esp_con > u_esp_sin:
            print(f"- La utilidad esperada CON seguro ({u_esp_con:.4f}) "
                  f"es MAYOR que SIN seguro ({u_esp_sin:.4f}).")
            print("  → Para un jugador averso al riesgo, pagar la prima")
            print("    es una decisión racional bajo estos parámetros.")
        elif u_esp_con < u_esp_sin:
            print(f"- La utilidad esperada CON seguro ({u_esp_con:.4f}) "
                  f"es MENOR que SIN seguro ({u_esp_sin:.4f}).")
            print("  → Con esta prima y cobertura, el seguro NO es racional")
            print("    para este jugador (prefiere asumir el riesgo).")
        else:
            print("- La utilidad esperada es prácticamente igual con y sin seguro.")
            print("  → El jugador es indiferente: seguro 'justo' en términos de utilidad.")

        # Comentario sobre capital esperado
        if cap_esp_con < cap_esp_sin:
            print("\n- Nota: El capital esperado monetario con seguro es menor,")
            print("        pero aun así puede ser racional contratarlo si la utilidad")
            print("        (que refleja aversión al riesgo) resulta mayor.")
        print(barra + "\n")


# =====================================================
#                       MAIN
# =====================================================

def main() -> None:
    # ----- PARÁMETROS POR DEFECTO (puedes modificarlos) -----
    capital_inicial = 100.0
    apuesta = 10.0
    prob_ganar = 0.45          # Juego ligeramente desfavorable
    tipo_utilidad = "sqrt"     # "lineal", "log" o "sqrt"

    prima = 5.0                # Prima del seguro
    cobertura = 0.5            # El seguro cubre el 50% de la pérdida

    # Crear jugador y seguro
    jugador = Jugador(
        capital_inicial=capital_inicial,
        apuesta=apuesta,
        prob_ganar=prob_ganar,
        tipo_utilidad=tipo_utilidad
    )

    seguro = Seguro(
        prima=prima,
        cobertura=cobertura
    )

    # Simulación
    simulacion = SimulacionRuina(
        jugador=jugador,
        seguro=seguro,
        max_rondas=100,
        n_simulaciones=5000
    )

    simulacion.comparar_sin_y_con_seguro()


if __name__ == "__main__":
    main()


               SIMULACIÓN DE RUINA CON Y SIN SEGURO         
Capital inicial:              100.00
Apuesta por ronda:             10.00
Prob. de ganar por ronda:     45.00%
Función de utilidad:      sqrt
N° de simulaciones:       5000
Máx. rondas por juego:    100

>>> ESCENARIO SIN SEGURO
Probabilidad de ruina:       66.8000%
Capital esperado final:         33.64
Utilidad esperada final:   -6676.8038

>>> ESCENARIO CON SEGURO
Prima del seguro:                5.00
Cobertura de pérdidas:         50.00%
Probabilidad de ruina:        0.2400%
Capital esperado final:        268.81
Utilidad esperada final:      16.2099

                           ANÁLISIS DE DECISIÓN             
- El seguro REDUCE la probabilidad de ruina de 66.8000% a 0.2400%.
- La utilidad esperada CON seguro (16.2099) es MAYOR que SIN seguro (-6676.8038).
  → Para un jugador averso al riesgo, pagar la prima
    es una decisión racional bajo estos parámetros.

